In [ ]:
from transformers import pipeline

# -----------------------------
# Load LLM
# -----------------------------

llm = pipeline(
    "text-generation",
    model="gpt2"
)

# helper function
def ask(prompt, tokens=120):
    result = llm(prompt, max_new_tokens=tokens)
    return result[0]["generated_text"].strip()


# -----------------------------
# Prompt 1 – Extract complaints
# -----------------------------

def extract_complaints(review):

    prompt = f"""
    Customer review: "{review}"

    What are the main complaints in this review?
    Give a short list.
    """

    return ask(prompt)


# -----------------------------
# Prompt 2 – Decide response actions
# -----------------------------

def decide_actions(complaints):

    prompt = f"""
    Given these customer complaints:

    {complaints}

    What should the company apologize for and what solution should be offered?
    """

    return ask(prompt)


# -----------------------------
# Prompt 3 – Generate reply email
# -----------------------------

def generate_email(actions):

    prompt = f"""
    Write a polite customer support reply email.

    Include these actions:
    {actions}

    The email should be professional and empathetic.
    """

    return ask(prompt, tokens=200)


# -----------------------------
# Prompt Chaining Controller
# -----------------------------

def prompt_chain(review):

    print("\nCustomer Review:")
    print(review)

    # Prompt 1
    complaints = extract_complaints(review)
    print("\nStep 1 – Complaints Identified:")
    print(complaints)

    # Prompt 2
    actions = decide_actions(complaints)
    print("\nStep 2 – Suggested Actions:")
    print(actions)

    # Prompt 3
    email = generate_email(actions)
    print("\nStep 3 – Generated Reply Email:")
    print(email)

    return email


# -----------------------------
# Example Input
# -----------------------------

review = "The product arrived late and the box was broken. Very disappointed."

final_email = prompt_chain(review)

print("\nFinal Output Email:")
print(final_email)

###Zeroshot classification(NLP)

In [ ]:
import os
import requests

API_URL = "https://router.huggingface.co/hf-inference/models/facebook/bart-large-mnli"
headers = {
    "Authorization": f"Bearer api_key",
}

def query(payload):
    response = requests.post(API_URL, headers=headers, json=payload)
    return response.json()

output = query({
    "inputs": "Hi, I recently bought a device from your company but it is not working as advertised and I would like to get reimbursed!",
    "parameters": {"candidate_labels": ["refund", "legal", "faq"]},
})
print(output)

[{'label': 'refund', 'score': 0.8777873516082764}, {'label': 'faq', 'score': 0.10522671788930893}, {'label': 'legal', 'score': 0.016985908150672913}]


**Types of Prompt chaining**
* **zero-shot prompting**  : direct question we give and it give answer as what we excpect as positive and negative,here we are not using examples(sentiment analysis,question and answer like mcqs)
* **one-shot prompting** : here we mention one example along with prompt .here it will leads sometimes to hallucination risk .example : movie was not bad : it returns negative
* **Few-shot prompting** : here we mention few examples along with prompt
* **chain of thought prompting** : this we use in coding, reasoning and aptitude scenario. here examples are not required
   * example: this is for coding assessment write code for prime numbers and think step by step and give correct answer( here it will think step by step)
   * if we use zero shot prompting it will not think step by step
* **RAG based prompting** : assume ,context(relevant content),user query,answer-->structure
    * example: answer the question based on relevant context
        * context :{context}
        * question :{user prompt}
        * answer :
* **prompt chaining** : here we use multiple llms
    * we can do this in langchain
    * here we give prompt for the 1st llm and it give response ,then that response we use as prompt and give to 2nd llm and it give response and then we give this response as prompt to 3rd llm


In [ ]:
###ZERO-SHOT PROMPTING
Prompt:
"Classify the sentiment of the following text as Positive, Negative, or Neutral.
Text: 'The product arrived late and was damaged.'
sentiment:"

In [ ]:
import os
import requests

API_URL = "https://router.huggingface.co/v1/chat/completions"
headers = {
    "Authorization": f"Bearer api_key",
}

def query(payload):
    response = requests.post(API_URL, headers=headers, json=payload)
    return response.json()

response = query({
    "messages": [
        {
            "role": "user",
            "content": "Classify the sentiment as Positive, Negative, or Neutral in one word.\n\nText: 'The product arrived late and was completely damaged.'\nSentiment:"
        }
    ],
    "model": "deepseek-ai/DeepSeek-R1:novita"
})
print("Sentiment:", response["choices"][0]["message"]['content'])

Sentiment: Negative


In [ ]:
###ONE-SHOT PROMPTING
Prompt:
"Extract the person name and location from the text.

Example:
Text: 'Elon Musk visited Berlin last Monday.'
Output: Person: Elon Musk | Location: Berlin

Now extract:
Text: 'Sundar Pichai gave a speech in New York yesterday.'
Output:"

Model Output: Person: Sundar Pichai | Location: New York

In [ ]:
import os
import requests

API_URL = "https://router.huggingface.co/v1/chat/completions"
headers = {
    "Authorization": f"Bearer api_key",
}

def query(payload):
    response = requests.post(API_URL, headers=headers, json=payload)
    return response.json()

response = query({
    "messages": [
        {
            "role": "user",
            "content": "Extract the person name and location from the text.\n\nText:'Sundar Pichai gave a speech in New York yesterday.'\nOutput:"
        }
    ],
    "model": "deepseek-ai/DeepSeek-R1:novita"
})
print("Response:", response["choices"][0]["message"]['content'])

Response: **Person:** Sundar Pichai  
**Location:** New York


In [ ]:
### FEW-SHOT PROMPTING
# ── Example 1 ──
{"role": "user",      "content": "Text: 'The food was absolutely delicious!'"},
{"role": "assistant", "content": "Positive"},

# ── Example 2 ──
{"role": "user",      "content": "Text: 'The package arrived damaged.'"},
{"role": "assistant", "content": "Negative"},
# ── Example 3 ──
{"role": "user",      "content": "Text: 'The store opens at 9 AM.'"},
{"role": "assistant", "content": "Neutral"}

In [ ]:
import os
import requests

API_URL = "https://router.huggingface.co/v1/chat/completions"
headers = {
    "Authorization": f"Bearer api_key",
}

def query(payload):
    response = requests.post(API_URL, headers=headers, json=payload)
    return response.json()

response = query({
    "messages": [
        {
            "role": "system",
            "content": "You are a sentiment classifier. Reply with only one word: Positive, Negative, or Neutral. No explanation."
        },
        # ── Example 1 ──
        {"role": "user","content": "Text: 'The food was absolutely delicious!'"},
        {"role": "assistant", "content": "Positive"},

        # ── Example 2 ──
        {"role": "user","content": "Text: 'The package arrived damaged.'"},
        {"role": "assistant", "content": "Negative"},

        # ── Example 3 ──
        {"role": "user","content": "Text: 'The store opens at 9 AM.'"},
        {"role": "assistant", "content": "Neutral"},

        # ── Actual Query ──
        {"role": "user","content": "Text: 'The battery life on this phone is terrible.'"},
    ],
    "model": "deepseek-ai/DeepSeek-R1:novita"
})
print("Response:", response["choices"][0]["message"]["content"])

Response: Negative


In [ ]:
#Chain of thought prompting
'''Q: Is 7 prime?
A: Check divisibility from 2 to √7 ≈ 2.6.
   7 ÷ 2 = 3.5 (not whole).
   No divisors found.
   Answer: Yes, 7 is prime.'''


In [ ]:
import os
import requests

API_URL = "https://router.huggingface.co/v1/chat/completions"
headers = {
    "Authorization": f"Bearer api_key",
}

def query(payload):
    response = requests.post(API_URL, headers=headers, json=payload)
    return response.json()

response = query({
    "messages": [
        {
            "role": "user",
            "content": """ give exact code for prime number.Think step by step by checking divisibility from 2."""
        }
    ],
    "model": "deepseek-ai/DeepSeek-R1:novita"
})
print("Response:", response["choices"][0]["message"]['content'])

Response: To determine if a number is prime, we check divisibility starting from 2 up to the square root of the number. Here's the Python code:

```python
def is_prime(n):
    if n <= 1:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    max_divisor = int(n ** 0.5) + 1
    for d in range(3, max_divisor, 2):
        if n % d == 0:
            return False
    return True
```

**Step-by-Step Explanation:**
1. **Check for numbers ≤ 1**: These are not prime.
2. **Check if the number is 2**: The smallest prime number.
3. **Check even numbers > 2**: If divisible by 2, not prime.
4. **Check odd divisors up to √n**: Any divisor larger than √n would have a corresponding divisor smaller than √n, so checking up to √n is sufficient. Increment by 2 to skip even numbers.
